# Chapter 13 &mdash; Turing's Definition of Computation

**Concept 1 of the Chapter 13 decomposition:** *Turing's Definition of Computation, and the Entscheidungsproblem*

A human with a one-dimensional tape, finitely many symbols, and a state of mind &mdash; and the Entscheidungsproblem it answered.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-Turings-Definition/Concept-Turings-Definition.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
from jove.AnimateTM      import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Turing's 1936 model is a deliberate **abstraction of a person calculating**: a
one-dimensional **tape** divided into cells, a **finite** set of symbols, a **finite**
"state of mind", and the ability to read, write and move one cell at a time.

The finiteness is the substance. A human can only distinguish finitely many symbols at
a glance and hold finitely many states of mind; everything unbounded must live on the
paper.

It was built to answer Hilbert's **Entscheidungsproblem** &mdash; is there a mechanical
procedure deciding the truth of any first-order statement? To answer *no*, Turing first
had to say precisely what "mechanical procedure" means. The machine is that
definition.

## 2. Definitions

### A first machine: flip every bit

In [ ]:
Flip = md2mc('''TM
!! Flip every bit, then halt in F.  F has NO outgoing transitions, which
!! is how a Jove TM signals "halt here".
I : 0 ; 1 , R -> I
I : 1 ; 0 , R -> I
I : . ; . , S -> F
''')

# --- thin wrappers over Jove's TM runner --------------------------------
# run_tm(T, tape, fuel) returns (truncated-paths, haltList).  A TM HALTS
# when no transition applies, and ACCEPTS if it halts in a final state.
# So an accepting state must have NO outgoing transitions, or the machine
# will run on past it.
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

def tm_tape(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return [cfg[2].rstrip('.') for cfg, _ in halts]

### The parts of Turing's abstraction

In [ ]:
PARTS = [("tape",      "one-dimensional, unbounded, divided into cells"),
         ("symbols",   "a FINITE alphabet -- Gamma, including the blank"),
         ("state",     "a FINITE 'state of mind' -- Q"),
         ("head",      "reads one cell, writes one cell, moves L, R or S"),
         ("program",   "Delta: (state, symbol) -> (state, symbol, direction)")]

## 3. Tests

The five parts, and where each lives in the dictionary.

In [ ]:
for name, what in PARTS:
    print("  %-8s %s" % (name, what))
print()
for k in ['Q', 'Sigma', 'Gamma', 'q0', 'B', 'F']:
    v = Flip[k]
    print("%-6s : %s" % (k, sorted(v) if isinstance(v, set) else v))

The machine runs, and the tape is the only unbounded thing.

In [ ]:
print("states :", sorted(Flip["Q"]), " -- finite")
print("symbols:", sorted(Flip["Gamma"]), " -- finite")
print()
for t in ['0101', '111', '0']:
    print("  %-8r -> %s" % (t, tm_tape(Flip, t)))
assert tm_tape(Flip, '0101') == ['1010']

It halts in `F`, which has no outgoing transitions.

In [ ]:
outgoing = [k for k in Flip["Delta"] if k[0] == 'F']
print("transitions out of F :", outgoing)
assert not outgoing
assert tm_accepts(Flip, '0101')
print("\nA Jove TM halts when STUCK.  An accepting state must be a dead end.")

Longer tapes cost more steps, not more states.

In [ ]:
for n in [2, 4, 8, 16]:
    t = '01' * n
    out = tm_tape(Flip, t, fuel=4*len(t) + 20)
    print("  |tape| = %2d -> %s" % (len(t), out[0][:20] + ('...' if len(t) > 20 else '')))
    assert out[0] == t.replace('0', 'x').replace('1', '0').replace('x', '1')
print("\n|Q| = %d throughout." % len(Flip["Q"]))

The Entscheidungsproblem, in one line.

In [ ]:
print("Hilbert asked  : is there a mechanical procedure deciding first-order truth?")
print("Turing answered: no -- and here is what 'mechanical procedure' means.")
print()
print("The definition had to come first.  That is why the machine exists.")

## 4. Animation

The bit-flipper, animated: watch the head sweep right rewriting each cell.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateTM import *
AnimateTM(Flip, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Which part of the model is unbounded? Which parts are deliberately finite?
2. Why does Turing describe a *human* computer rather than a device?
3. Add a state that returns the head to the left end. How many extra transitions?

In [ ]:
# Your work for the exercises above.